# Merchant Feature Engineering — BNPL Project

**Goal:** Convert curated transaction Parts 2–4 into a reproducible, merchant-level feature table for ranking and segmentation.

**Important assumptions:** `dollar_value` represents gross transaction value (not BNPL fee revenue); fraud flags describe consumer/merchant risk rather than confirmed fraudulent orders; geographic variables describe **consumers'** areas, not merchant store locations. The notebook does not silently remove fraud-flagged orders, because the flags are not verified transaction-level fraud labels.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Works when launched from either the repository root or a notebooks/ subfolder.
ROOT = Path.cwd()
if not (ROOT / "data" / "curated" / "curated_transactions").exists():
    ROOT = ROOT.parent
INPUT_DIR = ROOT / "data" / "curated" / "curated_transactions"
OUTPUT_DIR = ROOT / "data" / "curated" / "merchant_features"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FILES = [INPUT_DIR / f"part_{i}.parquet" for i in (2, 3, 4)]
assert all(p.exists() for p in FILES), f"Missing input: {[str(p) for p in FILES if not p.exists()]}"
print('Repository:', ROOT)


Repository: /Users/amyliu/Desktop/MAST30034_Project_2


## 1. Read curated transactions
Read only columns used downstream to limit memory consumption. The three parts are transaction partitions, not three different types of features.

In [ ]:
import pyarrow.parquet as pq

CORE = [
    'merchant_abn', 'order_id', 'order_datetime', 'dollar_value',
    'consumer_id', 'name_merchant', 'tags', 'consumer_is_fraud',
    'merchant_is_fraud', 'state', 'SA2_CODE_2021',
    'disadvantage_score', 'economic_resources_score'
]

OPTIONAL = [
    'consumer_fraud_probability',
    'merchant_fraud_probability'
]

for p in FILES:
    parquet = pq.ParquetFile(p)
    available = set(parquet.schema.names)

    missing = set(CORE[:5]) - available
    if missing:
        raise ValueError(f"{p.name} missing required columns: {missing}")

    cols = [c for c in CORE + OPTIONAL if c in available]

    print(f"{p.name}: {parquet.metadata.num_rows:,} rows")
    print(f"Available columns: {len(cols)}")

part_2.parquet: 3,643,266 rows
Available columns: 15
part_3.parquet: 4,508,106 rows
Available columns: 15
part_4.parquet: 6,044,133 rows
Available columns: 15


In [ ]:

frames = [
    pd.read_parquet(p, columns=[
        "merchant_abn", "order_id", "order_datetime",
        "dollar_value", "consumer_id", "name_merchant",
        "tags", "consumer_is_fraud", "merchant_is_fraud",
        "state", "SA2_CODE_2021",
        "disadvantage_score", "economic_resources_score"
    ])
    for p in FILES
]

tx = pd.concat(frames, ignore_index=True)
del frames



KeyboardInterrupt: 

## 2. Validate and prepare transactions

Only exact repeated `order_id` values are candidates for deduplication. Conflicting records sharing an ID cause an error instead of being silently discarded. Missing merchant IDs, dates or amounts are excluded and counted. Zero/negative values are excluded from commercial activity metrics, with counts reported. No arbitrary high-value outlier cap is imposed.

In [3]:
initial_n = len(tx)
if tx['order_id'].isna().any():
    raise ValueError('Null order_id detected; review source before deduplication')
dup = tx.duplicated('order_id', keep=False)
if dup.any():
    # Compare all loaded fields, treating missing values as equivalent.
    unique_rows = tx.loc[dup].drop_duplicates()
    if unique_rows['order_id'].duplicated().any():
        raise ValueError('Conflicting rows with the same order_id; investigate before proceeding')
    print('Repeated identical order IDs:', f'{tx.duplicated("order_id").sum():,}')
    tx = tx.drop_duplicates('order_id').copy()
else:
    print('No repeated order IDs')
tx['order_datetime'] = pd.to_datetime(tx['order_datetime'], errors='coerce')
tx['dollar_value'] = pd.to_numeric(tx['dollar_value'], errors='coerce')
invalid_required = tx[['merchant_abn', 'order_datetime', 'dollar_value']].isna().any(axis=1)
print('Rows missing required fields:', f'{invalid_required.sum():,}')
tx = tx.loc[~invalid_required].copy()
nonpositive = tx['dollar_value'] <= 0
print('Nonpositive transaction values excluded:', f'{nonpositive.sum():,}')
tx = tx.loc[~nonpositive].copy()
print(f'Retained {len(tx):,} of {initial_n:,} input rows ({len(tx)/initial_n:.1%})')
assert tx['order_id'].is_unique
assert tx['merchant_abn'].notna().all()
assert tx['dollar_value'].gt(0).all()


NameError: name 'tx' is not defined

## 3. Derived transaction fields

Monthly periods are based on calendar months. `consumer_is_fraud` and `merchant_is_fraud` are **risk flags**, not confirmed fraudulent transactions. Null flags are left missing. We avoid adding them together because the two flags can overlap.

In [ ]:
tx['month'] = tx['order_datetime'].dt.to_period('M')
tx['date'] = tx['order_datetime'].dt.normalize()
for col in ['consumer_is_fraud', 'merchant_is_fraud']:
    if col in tx:
        tx[col] = tx[col].astype('boolean')
    else:
        tx[col] = pd.Series(pd.NA, index=tx.index, dtype='boolean')
tx['any_fraud_flag'] = tx[['consumer_is_fraud', 'merchant_is_fraud']].fillna(False).any(axis=1)
# An observed risk-flag rate is available only if at least one flag is known.
tx['fraud_flag_observed'] = tx[['consumer_is_fraud', 'merchant_is_fraud']].notna().any(axis=1)


## 4. Core merchant features

Merchant-level revenue opportunity, customer reach, activity and transaction-size metrics. Rates based on calendar span include inactive months; `active_months` separately captures observed trading activity.

In [ ]:
g = tx.groupby('merchant_abn', sort=False, observed=True)
features = g.agg(
    total_transaction_value=('dollar_value', 'sum'),
    transaction_count=('order_id', 'size'),
    unique_customers=('consumer_id', 'nunique'),
    avg_transaction_value=('dollar_value', 'mean'),
    median_transaction_value=('dollar_value', 'median'),
    std_transaction_value=('dollar_value', 'std'),
    max_transaction_value=('dollar_value', 'max'),
    first_transaction=('order_datetime', 'min'),
    last_transaction=('order_datetime', 'max'),
    active_days=('date', 'nunique'),
    active_months=('month', 'nunique'),
    consumer_flag_rate=('consumer_is_fraud', 'mean'),
    merchant_flag_rate=('merchant_is_fraud', 'mean'),
    any_flagged_transaction_count=('any_fraud_flag', 'sum'),
    fraud_flag_observed_count=('fraud_flag_observed', 'sum'),
).reset_index()
features['observed_span_days'] = (features['last_transaction'] - features['first_transaction']).dt.days + 1
features['observed_span_months'] = (
    (features['last_transaction'].dt.year - features['first_transaction'].dt.year) * 12
    + features['last_transaction'].dt.month - features['first_transaction'].dt.month + 1
)
features['transactions_per_active_month'] = features['transaction_count'] / features['active_months']
features['value_per_active_month'] = features['total_transaction_value'] / features['active_months']
features['transactions_per_calendar_month'] = features['transaction_count'] / features['observed_span_months']
features['value_per_calendar_month'] = features['total_transaction_value'] / features['observed_span_months']
features['transactions_per_customer'] = features['transaction_count'] / features['unique_customers'].replace(0, np.nan)
features['value_per_customer'] = features['total_transaction_value'] / features['unique_customers'].replace(0, np.nan)
features['active_day_share'] = features['active_days'] / features['observed_span_days']
features['transaction_value_cv'] = features['std_transaction_value'].fillna(0) / features['avg_transaction_value']
features['fraud_flag_coverage'] = features['fraud_flag_observed_count'] / features['transaction_count']
# Overall flagged share is only defined for rows with observed flags.
observed_flagged = tx.loc[tx['fraud_flag_observed']].groupby('merchant_abn')['any_fraud_flag'].sum()
features['observed_any_fraud_flag_rate'] = (
    features['merchant_abn'].map(observed_flagged).fillna(0) /
    features['fraud_flag_observed_count'].replace(0, np.nan)
)
print('Merchant rows:', f'{len(features):,}')


## 5. Monthly revenue trend and stability

Monthly revenue is reindexed to a common calendar window across all partitions, filling **genuinely observed months with no orders** as zero. Growth compares the last two complete months of the data window, and is missing if the previous month has no revenue. This is a descriptive growth measure, not a forecast.

In [ ]:
monthly = tx.groupby(['merchant_abn', 'month'], observed=True)['dollar_value'].sum().rename('monthly_value').reset_index()
months = pd.period_range(tx['month'].min(), tx['month'].max(), freq='M')
# Global first/last months can be partial. Prefer full interior months for growth.
complete_months = months[1:-1] if len(months) >= 4 else months
if len(complete_months) >= 2:
    previous_month, latest_month = complete_months[-2], complete_months[-1]
else:
    previous_month = latest_month = None
monthly_wide = monthly.pivot(index='merchant_abn', columns='month', values='monthly_value')
monthly_wide = monthly_wide.reindex(columns=months, fill_value=0).fillna(0)
monthly_stats = pd.DataFrame(index=monthly_wide.index)
monthly_stats['monthly_value_std'] = monthly_wide.std(axis=1, ddof=0)
monthly_stats['monthly_value_mean'] = monthly_wide.mean(axis=1)
monthly_stats['monthly_value_cv'] = (monthly_stats['monthly_value_std'] /
                                    monthly_stats['monthly_value_mean'].replace(0, np.nan))
monthly_stats['months_with_transactions'] = (monthly_wide > 0).sum(axis=1)
if previous_month is not None:
    prev = monthly_wide[previous_month]
    latest = monthly_wide[latest_month]
    monthly_stats['recent_month_value'] = latest
    monthly_stats['previous_month_value'] = prev
    monthly_stats['recent_month_growth'] = ((latest - prev) / prev.replace(0, np.nan))
else:
    monthly_stats['recent_month_value'] = np.nan
    monthly_stats['previous_month_value'] = np.nan
    monthly_stats['recent_month_growth'] = np.nan
features = features.merge(monthly_stats.reset_index(), on='merchant_abn', how='left', validate='one_to_one')
print('Growth comparison months:', previous_month, latest_month)
del monthly_wide, monthly_stats


## 6. Repeat customers and customer concentration

Repeat-customer share captures loyalty/engagement. Top-customer share measures dependence on a single buyer, which may increase business concentration risk.

In [ ]:
customer = tx.dropna(subset=['consumer_id']).groupby(['merchant_abn', 'consumer_id'], observed=True).agg(
    customer_orders=('order_id', 'size'), customer_value=('dollar_value', 'sum')
).reset_index()
customer['is_repeat_customer'] = customer['customer_orders'] >= 2
customer_stats = customer.groupby('merchant_abn', observed=True).agg(
    repeat_customer_share=('is_repeat_customer', 'mean'),
    largest_customer_value=('customer_value', 'max'),
).reset_index()
features = features.merge(customer_stats, on='merchant_abn', how='left', validate='one_to_one')
features['largest_customer_value_share'] = (features['largest_customer_value'] /
                                            features['total_transaction_value'])
del customer, customer_stats


## 7. Merchant descriptions, segments and customer geography

`tags` is kept as descriptive metadata for later segmentation, not assumed to be a clean category. `state` and `SA2_CODE_2021` refer to the customer's geography in the joined data, so these are **customer footprint** features, not merchant location features. Socioeconomic scores are customer-transaction-weighted proxies; they should not be interpreted as merchant-level socioeconomic attributes.

In [ ]:
def first_nonmissing(s):
    s = s.dropna()
    return s.iloc[0] if len(s) else None

# Descriptions may contain non-hashable list-like tags, so don't group by tags.
meta_cols = [c for c in ['name_merchant', 'tags'] if c in tx.columns]
if meta_cols:
    meta = g[meta_cols].agg(first_nonmissing).reset_index()
    features = features.merge(meta, on='merchant_abn', how='left', validate='one_to_one')
if 'state' in tx.columns:
    state_count = tx.groupby('merchant_abn')['state'].nunique().rename('unique_customer_states')
    features['unique_customer_states'] = features['merchant_abn'].map(state_count)
if 'SA2_CODE_2021' in tx.columns:
    sa2_count = tx.groupby('merchant_abn')['SA2_CODE_2021'].nunique().rename('unique_customer_sa2s')
    features['unique_customer_sa2s'] = features['merchant_abn'].map(sa2_count)
for col in ['disadvantage_score', 'economic_resources_score']:
    if col in tx.columns:
        score = pd.to_numeric(tx[col], errors='coerce')
        avg = score.groupby(tx['merchant_abn']).mean()
        features['customer_mean_' + col] = features['merchant_abn'].map(avg)


## 8. Validation and export

Check unique merchant identifiers, transaction reconciliation, and feature ranges. The output contains **all observed merchants**, including those with very limited history; a later ranking notebook can apply transparent eligibility or confidence rules.

In [ ]:
assert features['merchant_abn'].is_unique
assert features['transaction_count'].sum() == len(tx)
assert np.isclose(features['total_transaction_value'].sum(), tx['dollar_value'].sum(), rtol=1e-10)
assert features['transaction_count'].ge(1).all()
assert features['unique_customers'].ge(0).all()
assert features['active_months'].ge(1).all()
assert features['largest_customer_value_share'].dropna().between(0, 1.0000001).all()
assert features['repeat_customer_share'].dropna().between(0, 1).all()
assert features['fraud_flag_coverage'].between(0, 1).all()
print('VALIDATION PASSED')
print('Merchants:', f'{len(features):,}')
print('Total transactions:', f'{features["transaction_count"].sum():,}')
print('Value:', f'${features["total_transaction_value"].sum():,.2f}')
print('Fraud coverage (transaction-weighted):', f'{tx["fraud_flag_observed"].mean():.2%}')
print('Missing growth:', f'{features["recent_month_growth"].isna().mean():.1%}')
print('Top missing feature counts:')
print(features.isna().sum().sort_values(ascending=False).head(12).to_string())
OUTPUT = OUTPUT_DIR / 'merchant_features.parquet'
features.to_parquet(OUTPUT, index=False)
print('Saved:', OUTPUT)
display(features[['merchant_abn', 'total_transaction_value', 'transaction_count',
                  'unique_customers', 'transactions_per_active_month',
                  'recent_month_growth', 'fraud_flag_coverage']].head())


## 9. Quick exploratory plots and ranking handoff

The log-scale plot makes the long-tailed merchant value distribution more interpretable. The next notebook should normalize skewed features, apply transparent scoring weights, assess fraud-data coverage and conduct sensitivity analysis before producing Top 100 and segment Top 10 lists.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(np.log1p(features['total_transaction_value']), bins=40)
axes[0].set(title='Merchant transaction value (log1p)', xlabel='log(1 + transaction value)', ylabel='Merchants')
axes[1].scatter(np.log1p(features['unique_customers']),
                np.log1p(features['total_transaction_value']), s=7, alpha=0.3)
axes[1].set(title='Customer reach vs transaction value', xlabel='log(1 + unique customers)',
            ylabel='log(1 + transaction value)')
plt.tight_layout()
plt.show()
